# Chapter 21 — Linear Function Approximation

In tabular RL, we store one value per state. When the state space is large or continuous,  
we need **function approximation** — parameterizing V or Q with a compact set of weights.

This notebook covers:
1. Linear function approximation: V(s) = **w** · φ(s)
2. Feature vectors for gridworlds
3. Semi-gradient TD(0) with linear FA
4. Comparing tabular vs. linear FA
5. Visualizing the learned value surface

In [ ]:
import numpy as np
import random

---
## Part 1 — Linear Function Approximation

The value function is approximated as:

$$\hat{V}(s, \mathbf{w}) = \mathbf{w}^\top \phi(s) = \sum_i w_i \phi_i(s)$$

Where:
- **w** ∈ ℝᵈ — learnable weight vector (d << |S|)
- **φ(s)** ∈ ℝᵈ — feature vector encoding state s

Linear FA generalizes across states that share similar features.

In [ ]:
def linear_value(w, phi_s):
    """Compute V(s) = w · phi(s)."""
    return np.dot(w, phi_s)

# Demo: 3-feature weight vector
w = np.array([0.5, -0.2, 0.8])
phi_s1 = np.array([1.0, 0.5, 0.3])
phi_s2 = np.array([0.0, 1.0, 0.1])

print(f"V(s1) = w · phi(s1) = {linear_value(w, phi_s1):.4f}")
print(f"V(s2) = w · phi(s2) = {linear_value(w, phi_s2):.4f}")

print()
print("Key property: ∂V/∂w = phi(s)")
print("The gradient of V w.r.t. w is simply the feature vector!")
print("This makes the update rule simple: w += alpha * delta * phi(s)")

---
## Part 2 — Feature Vectors for a Gridworld

For an N×N gridworld with state (row, col), we can create features:

| Feature | Formula | Captures |
|---------|---------|----------|
| row | row/N | Position row |
| col | col/N | Position col |
| row² | (row/N)² | Nonlinear row |
| col² | (col/N)² | Nonlinear col |
| row×col | (row/N)(col/N) | Interaction |
| bias | 1.0 | Intercept |

In [ ]:
def make_features(row, col, grid_size=5):
    """Feature vector phi(s) for gridworld state (row, col)."""
    r = row / grid_size
    c = col / grid_size
    return np.array([
        1.0,      # bias
        r,        # row
        c,        # col
        r**2,     # row squared
        c**2,     # col squared
        r * c,    # interaction
    ])

# Show feature vectors for a few states in a 5x5 grid
print("Feature vectors for a 5x5 gridworld:")
feature_names = ['bias', 'row', 'col', 'row²', 'col²', 'row×col']
print(f"{'State':<12}", "  ".join(f"{n:>8}" for n in feature_names))
for r, c in [(0,0), (0,4), (2,2), (4,4), (1,3)]:
    phi = make_features(r, c, 5)
    print(f"({r},{c}):      ", "  ".join(f"{v:>8.4f}" for v in phi))

---
## Part 3 — Semi-Gradient TD(0) with Linear FA

The semi-gradient TD(0) update for weight vector **w**:

$$\delta = R + \gamma \hat{V}(s', \mathbf{w}) - \hat{V}(s, \mathbf{w})$$

$$\mathbf{w} \leftarrow \mathbf{w} + \alpha \, \delta \, \nabla_\mathbf{w} \hat{V}(s, \mathbf{w}) = \mathbf{w} + \alpha \, \delta \, \phi(s)$$

It's called "semi-gradient" because we **stop the gradient** at the bootstrap target  
(we don't differentiate through V(s')).

In [ ]:
def semi_gradient_td_update(w, phi_s, phi_s_next, reward, alpha, gamma, done):
    """Semi-gradient TD(0) update for linear function approximation.
    
    w:           weight vector (modified in place)
    phi_s:       feature vector of current state s
    phi_s_next:  feature vector of next state s' (ignored if done)
    reward:      R received on this transition
    alpha:       learning rate
    gamma:       discount factor
    done:        True if s' is terminal
    
    Returns: (updated w, td_error)
    """
    V_s = np.dot(w, phi_s)
    V_s_next = 0.0 if done else np.dot(w, phi_s_next)
    td_error = reward + gamma * V_s_next - V_s
    # Gradient of V(s) w.r.t. w is phi_s (for linear FA)
    w += alpha * td_error * phi_s
    return w, td_error

# Quick test
w = np.zeros(6)
phi_A = make_features(0, 0, 5)
phi_B = make_features(0, 1, 5)

w, delta = semi_gradient_td_update(w.copy(), phi_A, phi_B, reward=0.0, alpha=0.1, gamma=0.9, done=False)
print(f"After 1 update (reward=0, from (0,0) to (0,1)):")
print(f"  TD error: {delta:.4f}")
print(f"  w: {w}")
print()
print("Note: w unchanged (delta=0 since V(s)=V(s')=0 and reward=0)")

---
## Part 4 — Train Linear FA on a Gridworld

Set up a simple 5×5 gridworld with a goal at (4,4).  
Train semi-gradient TD(0) and visualize the learned value surface.

In [ ]:
# Gridworld environment
GRID_SIZE = 5
GOAL = (4, 4)

def grid_step(state, action):
    """action: 0=up, 1=down, 2=left, 3=right"""
    r, c = state
    dr, dc = [(-1,0),(1,0),(0,-1),(0,1)][action]
    new_r = max(0, min(GRID_SIZE-1, r + dr))
    new_c = max(0, min(GRID_SIZE-1, c + dc))
    s_prime = (new_r, new_c)
    done = s_prime == GOAL
    reward = 1.0 if done else -0.01
    return s_prime, reward, done

def random_start():
    while True:
        s = (random.randint(0, GRID_SIZE-1), random.randint(0, GRID_SIZE-1))
        if s != GOAL:
            return s

# Train
random.seed(42)
np.random.seed(42)
w = np.zeros(6)
alpha, gamma = 0.01, 0.99
n_episodes = 2000

td_errors = []
for ep in range(n_episodes):
    s = random_start()
    ep_error = 0.0
    for step in range(100):
        action = random.randint(0, 3)
        s_prime, r, done = grid_step(s, action)
        phi_s = make_features(*s, GRID_SIZE)
        phi_sp = make_features(*s_prime, GRID_SIZE)
        w, delta = semi_gradient_td_update(w, phi_s, phi_sp, r, alpha, gamma, done)
        ep_error += abs(delta)
        if done:
            break
        s = s_prime
    td_errors.append(ep_error)

print("Training complete.")
print(f"Final weights: {w}")
print(f"Mean TD error (last 100 eps): {sum(td_errors[-100:])/100:.4f}")

---
## Part 5 — Visualize the Learned Value Surface

In [ ]:
# Compute learned V for every grid state
V_learned = np.zeros((GRID_SIZE, GRID_SIZE))
for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        phi = make_features(r, c, GRID_SIZE)
        V_learned[r, c] = np.dot(w, phi)

# Set goal value
V_learned[GOAL] = 0.0

print("Learned Value Function (Linear FA):")
print("(Goal is at bottom-right corner (4,4))")
print()
# Print as grid (flip rows for visual — row 0 at top)
for r in range(GRID_SIZE):
    row_str = ""
    for c in range(GRID_SIZE):
        if (r,c) == GOAL:
            row_str += "  GOAL "
        else:
            row_str += f" {V_learned[r,c]:+.3f}"
    print(row_str)

print()
print("Values should increase toward (4,4).")
print(f"V(0,0) = {V_learned[0,0]:.4f}  (far from goal, lower value)")
print(f"V(3,3) = {V_learned[3,3]:.4f}  (close to goal, higher value)")

---
## Part 6 — Compare Tabular vs. Linear FA

In [ ]:
# Tabular TD(0) for comparison
random.seed(42)
V_tabular = {(r,c): 0.0 for r in range(GRID_SIZE) for c in range(GRID_SIZE)}
alpha_tab = 0.1

for ep in range(n_episodes):
    s = random_start()
    for step in range(100):
        action = random.randint(0, 3)
        s_prime, r, done = grid_step(s, action)
        V_sp = 0.0 if done else V_tabular[s_prime]
        td_err = r + gamma * V_sp - V_tabular[s]
        V_tabular[s] += alpha_tab * td_err
        if done:
            break
        s = s_prime

print("Comparison: Tabular TD(0) vs. Linear FA TD(0)")
print(f"{'State':<10} {'Tabular':>10} {'Linear FA':>10}")
print("-" * 32)
for r in range(0, GRID_SIZE, 2):
    for c in range(0, GRID_SIZE, 2):
        if (r,c) == GOAL:
            continue
        v_tab = V_tabular[(r,c)]
        v_lin = V_learned[r,c]
        print(f"({r},{c}):      {v_tab:>10.4f} {v_lin:>10.4f}")

n_params_tabular = GRID_SIZE * GRID_SIZE
n_params_linear  = len(w)
print(f"\nTabular parameters: {n_params_tabular}")
print(f"Linear FA parameters: {n_params_linear}")
print(f"Compression ratio: {n_params_tabular / n_params_linear:.1f}x")

---
## Part 7 — Exercise: Tile Coding Feature Vector

**Tile coding** divides the state space into overlapping grids (tiles) and uses binary features.  
Implement a simple 1D tile coding.

In [ ]:
def tile_coding_1d(x, n_tilings=4, n_tiles=8, x_min=0.0, x_max=1.0):
    """Simple 1D tile coding.
    Returns binary feature vector of length n_tilings * n_tiles.
    Each tiling is offset by (tile_width / n_tilings) from the last.
    """
    features = np.zeros(n_tilings * n_tiles)
    tile_width = (x_max - x_min) / n_tiles
    offset_step = tile_width / n_tilings

    for tiling in range(n_tilings):
        offset = tiling * offset_step
        tile_idx = int((x - x_min + offset) / tile_width)
        tile_idx = max(0, min(n_tiles - 1, tile_idx))
        features[tiling * n_tiles + tile_idx] = 1.0

    return features

# Test: x=0.25 with 4 tilings of 8 tiles
phi = tile_coding_1d(0.25)
print(f"Feature vector for x=0.25: {phi.astype(int)}")
print(f"Number of active features: {int(phi.sum())}  (should be n_tilings = 4)")
print(f"Total features: {len(phi)}")

# Verify: nearby states share tiles (generalization)
phi1 = tile_coding_1d(0.25)
phi2 = tile_coding_1d(0.27)
phi3 = tile_coding_1d(0.50)
print(f"\nOverlap (0.25, 0.27): {int(np.dot(phi1, phi2))} shared tiles (nearby)")
print(f"Overlap (0.25, 0.50): {int(np.dot(phi1, phi3))} shared tiles (far)")

---
## Part 8 — Exercise: Semi-Gradient TD with Tile Coding

Run semi-gradient TD(0) on the 1D random walk problem using tile coding.

In [ ]:
# 1D random walk: positions in [0,1], terminal at 0 and 1
random.seed(7)
np.random.seed(7)

n_features = 4 * 10  # 4 tilings, 10 tiles each
w_tc = np.zeros(n_features)
alpha_tc = 0.1 / 4   # alpha / n_tilings (standard normalization)

def rw_step(x):
    step = random.choice([-0.05, 0.05])
    x_new = x + step
    done = x_new <= 0.0 or x_new >= 1.0
    r = 1.0 if x_new >= 1.0 else (-1.0 if x_new <= 0.0 else 0.0)
    return max(0.0, min(1.0, x_new)), r, done

for ep in range(1000):
    x = 0.5
    for _ in range(500):
        phi_x = tile_coding_1d(x, n_tilings=4, n_tiles=10)
        x_next, r, done = rw_step(x)
        phi_xn = tile_coding_1d(x_next, n_tilings=4, n_tiles=10)
        w_tc, _ = semi_gradient_td_update(w_tc, phi_x, phi_xn, r, alpha_tc, gamma=1.0, done=done)
        if done:
            break
        x = x_next

print("Learned V (random walk with tile coding):")
print(f"{'Position':>10}  {'V(x)':>8}  {'True V(x)':>10}")
for pos in np.linspace(0.05, 0.95, 10):
    phi = tile_coding_1d(pos, n_tilings=4, n_tiles=10)
    v_learned = np.dot(w_tc, phi)
    v_true = 2 * pos - 1   # true V(x) = x for 0-1 normalized; true is linear
    print(f"{pos:>10.2f}  {v_learned:>8.4f}  {v_true:>10.4f}")

---
## Summary

| Concept | Key Formula |
|---------|-------------|
| Linear FA | V(s,w) = w⊤φ(s) |
| Gradient | ∇ᵥV(s,w) = φ(s) |
| TD error | δ = R + γV(s',w) − V(s,w) |
| Weight update | w ← w + α·δ·φ(s) |
| Tile coding | Binary sparse features, built-in generalization |

**Key insight:** Linear FA gives us generalization — updating V for one state also updates  
V for similar states (those with overlapping feature vectors). This is the key advantage over tabular methods for large state spaces.

**Next:** Chapter 22 — Neural Network Function Approximation (DQN).